In [2]:
import pandas as pd
import numpy as np
import warnings

In [3]:
# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('cars.csv')

COL_PRICE = 'price' 
COL_ODO = 'mileage_v2'      
COL_LOCATION = 'region_name'
COL_TEXT = 'subject'

In [4]:
# 2. CHUẨN HÓA GIÁ & ODO
df[COL_PRICE] = pd.to_numeric(df[COL_PRICE], errors='coerce')
df['price_million'] = df[COL_PRICE] / 1000000
df[COL_ODO] = pd.to_numeric(df[COL_ODO], errors='coerce')
df['price_million'] = df['price_million'].fillna(df['price_million'].median())
df[COL_ODO] = df[COL_ODO].fillna(df[COL_ODO].median())
df[COL_ODO] = df[COL_ODO].astype(int)

In [8]:
# 3. MAP 63 TỈNH THÀNH
def clean_location(val):
    if pd.isna(val):
        return 'Không xác định'
    val = str(val).strip()
    for prefix in ['TP ', 'Tp ', 'Thành phố ', 'Tỉnh ', 'TP. ', 'Tp. ']:
        if val.lower().startswith(prefix.lower()):
            val = val[len(prefix):].strip()     
    mapping = {
        'Hồ Chí Minh': 'Hồ Chí Minh', 'Hcm': 'Hồ Chí Minh',
        'Hà Nội': 'Hà Nội', 'Hn': 'Hà Nội',
        'Bria - Vũng Tàu': 'Bà Rịa - Vũng Tàu', 'Thừa Thiên Huế': 'Thừa Thiên - Huế'}
    return mapping.get(val, val)
df[COL_LOCATION] = df[COL_LOCATION].apply(clean_location)

In [9]:
# 4. TẠO CỜ NHỊ PHÂN NLP TỪ TEXT
text_series = df[COL_TEXT].astype(str).str.lower()
df['flag_chinh_chu'] = text_series.str.contains(r'chính chủ|1 chủ|một chủ|gia đình', regex=True).astype(int)
df['flag_bao_duong_hang'] = text_series.str.contains(r'bảo dưỡng hãng|lịch sử hãng|full lịch sử|bảo hành', regex=True).astype(int)

In [10]:
# 5. LỌC NGOẠI LAI (OUTLIERS) BẰNG IQR
def remove_outliers_iqr(dataframe, columns):
    df_out = dataframe.copy()
    for col in columns:
        Q1 = df_out[col].quantile(0.25)
        Q3 = df_out[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_out = df_out[(df_out[col] >= lower_bound) & (df_out[col] <= upper_bound)]
    return df_out
initial_len = len(df)
df = remove_outliers_iqr(df, ['price_million', COL_ODO])
final_len = len(df)

In [11]:
# 6. KIỂM TRA VÀ LƯU FILE
print('--- KẾT QUẢ LỌC NGOẠI LAI BẰNG IQR ---')
print(f'Số dòng ban đầu: {initial_len}')
print(f'Số dòng sau khi lọc: {final_len}')
print(f'-> Đã loại bỏ: {initial_len - final_len} dòng ngoại lai có giá hoặc ODO quá bất thường.\n')
print('--- KẾT QUẢ TẠO CỜ NLP (5 DÒNG ĐẦU) ---')
display(df[[COL_TEXT, 'flag_chinh_chu', 'flag_bao_duong_hang']].head())
output_file = 'cars_cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'\nĐã lưu file hoàn chỉnh: {output_file}')


--- KẾT QUẢ LỌC NGOẠI LAI BẰNG IQR ---
Số dòng ban đầu: 15000
Số dòng sau khi lọc: 13614
-> Đã loại bỏ: 1386 dòng ngoại lai có giá hoặc ODO quá bất thường.

--- KẾT QUẢ TẠO CỜ NLP (5 DÒNG ĐẦU) ---


,subject,flag_chinh_chu,flag_bao_duong_hang
0,ISUZU DMAX NHẬP THÁI SỐ TỰ ĐỘNG SIÊU KENG RẤT ĐẸP,0,0
1,Ranger XLS AT,0,0
2,terano ll máy dầu 7 chỗ diezen 2 cầu tubo xe đẹp,0,0
3,BMW X6 2010 Full Đồ Chơi Của Hãng,0,0
4,Toyota Fortuner 4x2 2016 một chủ,1,0



Đã lưu file hoàn chỉnh: cars_cleaned.csv
